# 🤖 Huấn Luyện & Đánh Giá PPO 9D Cho Robot Tự Cân Bằng (Sim-to-Real STM32)

**Cấu hình đồng bộ 100% với Firmware `Project_self` (STM32L476RG):**
- **Chu kỳ điều khiển:** 200 Hz (`Ts = 5 ms`), Encoder cập nhật mỗi `20 ms` (50 Hz, 1400 CPR, bộ lọc vận tốc $N=10$).
- **Chuỗi truyền động:** Hệ số khuếch đại PPO `1.30x` → Bộ lọc PWM `0.60 × u_raw + 0.40 × u_prev` → Làm tròn số nguyên → Bù vùng chết mềm `80 PWM` (deadzone `|u| ≤ 2 → 0`).
- **Không gian quan sát 9D:** $[\theta_t, \dot{\theta}_t, x_t, v_t, \theta_{t-1}, \dot{\theta}_{t-1}, x_{t-1}, v_{t-1}, a_{t-1}]$.
- **Tiêu chí đánh giá & chọn checkpoint:** Xếp hạng theo thứ tự nghiêm ngặt **Ít ngã nhất → Nhiều ca giữ vị trí đạt nhất (`|θ| ≤ 0.5°`, `|x| ≤ 2 cm` trong trọn 2s cuối) → Sai số vị trí cuối thấp nhất**.

---
### Bước 1: Lấy code mới nhất từ GitHub & Cài đặt thư viện

In [ ]:
import os

if os.path.exists('configs'):
    !git fetch origin
    !git reset --hard origin/p1-balance-robot
elif os.path.exists('self_blancing'):
    %cd self_blancing
    !git fetch origin
    !git reset --hard origin/p1-balance-robot
else:
    !git clone -b p1-balance-robot https://github.com/nguyenthanhsang2306/self_blancing.git
    %cd self_blancing

!pip install -q gymnasium stable-baselines3 torch matplotlib pyyaml tensorboard

---
### Bước 2: Train 2 phiên bản `w_pos = 0.15` và `w_pos = 0.30` (giữ nguyên lọc PWM 0.60/0.40)

- Mỗi lần train sẽ xóa `best_policy_*.npz` cũ và **chỉ chọn checkpoint tốt nhất trong chính lần train đó** theo thứ tự: *Ít ngã nhất → Giữ vị trí 2s cuối đạt nhiều nhất → Sai số cuối nhỏ nhất*.

In [ ]:
!python train.py --config configs/default.yaml --w-pos 0.15 --tag wpos_015
!python train.py --config configs/default.yaml --w-pos 0.30 --tag wpos_030

---
### Bước 3: So sánh `w_pos = 0.15` vs `0.30` trên cùng 96 kịch bản (10s + 20–30s) & Chọn bản thắng cuộc

- Chạy trên cùng bộ seed cố định gồm **90 ca 10s** (`±1°, ±3°, ±5°` | nhiễu `0, ±1, ±2 N` | pin `7.2–8.4 V`) và **6 ca dài 20–30s** để bắt lỗi trôi chậm.
- Tự động lưu bản thắng cuộc sang `outputs/checkpoints/best_policy.npz` và xuất `outputs/firmware_headers/rl_sb3_stm32.h`.

In [ ]:
!python inference.py --compare outputs/checkpoints/best_policy_wpos_015.npz outputs/checkpoints/best_policy_wpos_030.npz --export-winner

---
### Bước 4: Vẽ đồ thị kiểm tra bản thắng cuộc (10s chuẩn, 25s bắt trôi chậm, và va đập 1.5 N)

In [ ]:
from IPython.display import Image, display

print('--- 1) Đáp ứng chuẩn 10s (góc ban đầu seed=42) ---')
!python inference.py --duration 10.0
display(Image(filename='outputs/figures/rl_inference_test.png'))

print('--- 2) Kiểm tra trôi chậm 25s (góc ban đầu +5.0°) ---')
!python inference.py --angle 5.0 --duration 25.0
display(Image(filename='outputs/figures/rl_inference_test.png'))

print('--- 3) Kiểm tra va đập ngoại lực +1.5 N tại t = 2.0s ---')
!python inference.py --angle 0.0 --kick 1.5 --duration 10.0
display(Image(filename='outputs/figures/rl_inference_test.png'))

---
### Bước 5: Kiểm tra sai số toán học xuất C Header (`PyTorch ↔ Numpy ↔ C`)

In [ ]:
!python verify_export.py

---
### Bước 6: Tải file `rl_sb3_stm32.h` về máy để chép vào `Project_self/Core/Inc/`

> **Lưu ý:** Nếu trình duyệt chặn tải tự động, mở biểu tượng thư mục 📁 ở cột trái Colab → `self_blancing/outputs/firmware_headers/rl_sb3_stm32.h` → chuột phải chọn **Download**.

In [ ]:
!zip -q -r rl_balancing_results.zip outputs/

from google.colab import files
try:
    files.download('outputs/firmware_headers/rl_sb3_stm32.h')
    files.download('rl_balancing_results.zip')
except Exception as e:
    print(f'Tự động tải bị chặn ({e}). Hãy tải thủ công từ cột trái Colab: outputs/firmware_headers/rl_sb3_stm32.h')